# BIST asistanı: LoRA eğitimi v2 (Colab, ücretsiz T4)

Hücreleri **yukarıdan aşağı sırayla** çalıştır (hücrenin solundaki ▶ düğmesi ya da Shift+Enter).

Başlamadan önce:
1. Menü: **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU → Kaydet**.
2. Google Drive'ında **`MyDrive/BIST_v2`** klasöründe `train_gpu.py` ve `tokenized.jsonl.gz` olmalı.

Eğitim sürerken bu sekme açık kalsın ve Mac uykuya geçmesin. Mac'te başka işler yapabilirsin.
Adaptör her 25 adımda Drive'a kaydedilir. Oturum koparsa **1, 2, 3 ve 5.** hücreleri yeniden çalıştır, eğitim kaldığı yerden sürer.

### 1. GPU kontrolü

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU yok: Çalışma zamanı > Çalışma zamanı türünü değiştir > T4 GPU seç, sonra bu hücreyi yeniden çalıştır."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0), "|",
      round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

### 2. Google Drive'ı bağla
Açılan pencerede Google hesabını seçip izin ver.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
W = '/content/drive/MyDrive/BIST_v2'
for f in ('train_gpu.py', 'tokenized.jsonl.gz'):
    assert os.path.exists(f'{W}/{f}'), f"{W}/{f} yok: paketi Drive'da MyDrive/BIST_v2 klasörüne yükle."
print('paket bulundu:', sorted(os.listdir(W)))

### 3. Kütüphaneler (Mac'teki sürümlerle aynı, ~1 dk)

In [ ]:
!pip install -q "transformers==5.24.0" "peft==0.22.0" "accelerate==1.16.0"
!python -c "import transformers, peft; print('transformers', transformers.__version__, '| peft', peft.__version__)"

### 4. Kısa deneme (yalnızca ilk seferde, ~5-10 dk)
Model indirilir (~8 GB) ve en uzun 2 konuşma dahil 6 konuşmayla eğitim denenir.
Sonunda `"kind": "done"` satırı görünmeli; `loss` değerleri sayı olmalı (NaN değil). Çıktının ekran görüntüsünü at.

In [ ]:
!cd "{W}" && python train_gpu.py train --smoke --out /content/smoke_adapter --log /content/smoke_log.jsonl

### 5. Asıl eğitim (tahmini 2-4 saat)
Her 5 adımda bir satır yazar: `eta_min` = kalan dakika, `peak_gb` = GPU belleği.
Oturum koparsa 1, 2, 3 ve bu hücreyi yeniden çalıştır (`--resume`: Drive'daki son kayıttan sürer).

In [ ]:
!cd "{W}" && python train_gpu.py train --resume --out "{W}/adapter_v2" --log "{W}/train_v2_log.jsonl"

### 6. Paketle
Bitince bu hücreyi çalıştır, sonra Drive'daki `BIST_v2/adapter_v2.zip` dosyasını Mac'e indir (~60 MB).

In [ ]:
!cd "{W}" && zip -qr adapter_v2.zip adapter_v2 train_v2_log.jsonl && ls -la adapter_v2.zip